## 1. Imports

The notebook uses pandas/NumPy for data preparation, scikit-learn for splitting, scaling, and evaluation, and TensorFlow/Keras for the neural network.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("TensorFlow version:", tf.__version__)

## 2. Reproducibility

A fixed seed makes the experiment easier to reproduce.

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Task 1 — Data Ingestion, Cleaning & Preprocessing

## 3. Download and Load the Dataset

The dataset is downloaded directly from Kaggle using `kagglehub`. The code searches the downloaded folder for the Telco CSV so it does not depend on a hard-coded local filename.

In [ ]:
import kagglehub
from pathlib import Path

# Kaggle dataset:
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Path to dataset files:", path)

# The Kaggle dataset contains this CSV file:
# WA_Fn-UseC_-Telco-Customer-Churn.csv
csv_path = Path(path) / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# In case the downloaded version is nested, search for the exact file name.
if not csv_path.exists():
    matches = list(Path(path).rglob("WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    if not matches:
        raise FileNotFoundError(
            "WA_Fn-UseC_-Telco-Customer-Churn.csv was not found in the Kaggle download."
        )
    csv_path = matches[0]

df = pd.read_csv(csv_path)

print("CSV file:", csv_path)
print("Data shape:", df.shape)
df.head()

In [ ]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Target column:", "Churn")
print("\nColumn names:")
print(df.columns.tolist())

## 4. Inspect the Raw Data

Before preprocessing, check the column names, data types, and missing values.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

## 5. Clean `TotalCharges`

The Telco dataset can contain whitespace entries in `TotalCharges`. We convert the column to numeric using `pd.to_numeric(..., errors='coerce')`. Invalid/whitespace values become `NaN`, then they are filled with `0.0` as required.

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("TotalCharges missing values:", df["TotalCharges"].isnull().sum())
print("TotalCharges example values:")
print(df["TotalCharges"].head())

## 6. Drop `customerID` and Encode `Churn`

- `customerID` is an identifier, so it is removed.
- `Churn` is converted from `Yes/No` to `1/0`.

In [ ]:
df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)

print("Updated shape:", df.shape)
print("\nChurn values:")
print(df["Churn"].value_counts())

## 7. Convert Categorical Columns to Dummy Variables

Use `pd.get_dummies(..., drop_first=True)` to convert categorical text columns into indicator variables. `dtype=int` keeps the resulting feature matrix numeric.

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("\nNumber of preprocessed features:", X.shape[1])
print("X shape:", X.shape)
X.head()

## 8. Train/Test Split

Use an 80/20 split and stratify by the churn label so the class distribution is preserved in both partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest churn distribution:")
print(y_test.value_counts(normalize=True))

## 9. Normalize Continuous Numeric Attributes

Only the required continuous attributes are standardized:
- `tenure`
- `MonthlyCharges`
- `TotalCharges`

The scaler is fitted **only on the training partition** and then used to transform both training and test data.

In [ ]:
continuous_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[continuous_columns] = scaler.fit_transform(
    X_train[continuous_columns]
)

X_test[continuous_columns] = scaler.transform(
    X_test[continuous_columns]
)

print("Scaler fitted on training data only.")
print("\nScaled training columns:")
print(X_train[continuous_columns].head())

## 10. Convert Features to NumPy Arrays

Keras will receive a numeric matrix with shape `(samples, features)`.

In [ ]:
X_train = X_train.astype(np.float32).to_numpy()
X_test = X_test.astype(np.float32).to_numpy()
y_train = y_train.to_numpy().astype(np.float32)
y_test = y_test.to_numpy().astype(np.float32)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of input features:", X_train.shape[1])

# Task 2 — Keras Sequential Model Construction & Evaluation

## 11. Build the Binary Classification Neural Network

The architecture follows the task exactly:

1. Input layer matching the number of preprocessed features.
2. Hidden `Dense` layer with **16 units** and **ReLU** activation.
3. Output `Dense` layer with **1 unit** and **Sigmoid** activation.

This follows the `Sequential` and `Dense` structure used in the provided TensorFlow notebook.

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.summary()

## 12. Compile the Model

Required configuration:
- Optimizer: Adam
- Learning rate: `0.005`
- Loss: `binary_crossentropy`
- Metrics: accuracy and AUC

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.005
)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

## 13. Train the Network

Train for **50 epochs**, use a **batch size of 32**, and reserve **20% of the training partition** for validation.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

## 14. Training & Validation Loss Curves

The training history returned by `model.fit()` contains both training and validation loss.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Crossentropy Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Evaluate on the Held-Out Test Set

The test set was not used during model fitting. We evaluate it here after training.

In [ ]:
test_results = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_results[0])
print("Test Accuracy:", test_results[1])
print("Test ROC-AUC (Keras):", test_results[2])

## 16. Generate Churn Probabilities and Class Predictions

The sigmoid output is a churn probability. Using the required standard cutoff `τ = 0.50`, probabilities at or above 0.50 are assigned to class 1 (Churn).

In [ ]:
churn_probabilities = model.predict(
    X_test,
    verbose=0
).ravel()

threshold = 0.50
y_pred = (churn_probabilities >= threshold).astype(int)

print("First 10 churn probabilities:")
print(churn_probabilities[:10])

print("\nFirst 10 predicted classes:")
print(y_pred[:10])

print("\nFirst 10 actual classes:")
print(y_test[:10].astype(int))

## 17. Confusion Matrix, Precision, Recall & ROC-AUC

In [ ]:
cm = confusion_matrix(y_test, y_pred)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, churn_probabilities)

print("Confusion Matrix:")
print(cm)

print(f"\nTest Accuracy : {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall   : {recall:.4f}")
print(f"Test ROC-AUC  : {roc_auc:.4f}")

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
).plot()

plt.title("Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 18. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Churn", "Churn"],
        digits=4,
        zero_division=0
    )
)

## 19. Customer Factors Most Strongly Associated with Churn

To examine which preprocessed customer factors are most strongly correlated with the churn label, calculate Pearson correlation between each numeric/dummy feature and `Churn`.

**Important:** dummy-variable correlations refer to the encoded category indicators created with `drop_first=True`, so each indicator represents a category relative to its omitted reference category.

In [ ]:
correlation_df = X.copy()
correlation_df["Churn"] = y.values

churn_correlations = (
    correlation_df.corr(numeric_only=True)["Churn"]
    .drop("Churn")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Top 15 features by absolute correlation with Churn:")
print(churn_correlations.head(15))

In [ ]:
top_correlations = churn_correlations.head(10).sort_values()

plt.figure(figsize=(9, 6))
top_correlations.plot(kind="barh")
plt.xlabel("Correlation with Churn")
plt.ylabel("Feature")
plt.title("Top Customer Factors Associated with Churn")
plt.tight_layout()
plt.show()

# 20. Final Results Summary

The following cell prints the main deliverable metrics in one place.

In [ ]:
final_validation_loss = history.history["val_loss"][-1]

print("=" * 55)
print("FINAL TEST RESULTS")
print("=" * 55)
print(f"Test Accuracy       : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Test ROC-AUC        : {roc_auc:.4f}")
print(f"Churn Precision     : {precision:.4f} ({precision * 100:.2f}%)")
print(f"Churn Recall        : {recall:.4f} ({recall * 100:.2f}%)")
print(f"Final Validation Loss: {final_validation_loss:.4f}")
print("=" * 55)

## Benchmark Check

The benchmark ranges in the task are targets for the completed pipeline. Exact results can vary slightly with TensorFlow/scikit-learn versions and training randomness. The cell below checks whether the current run falls inside the requested ranges.

In [ ]:
benchmarks = {
    "Test Accuracy": (0.790, 0.810, accuracy),
    "Test ROC-AUC": (0.835, 0.845, roc_auc),
    "Churn Recall": (0.500, 0.550, recall),
    "Churn Precision": (0.620, 0.670, precision),
    "Final Validation Loss": (0.410, 0.430, final_validation_loss),
}

for metric_name, (low, high, value) in benchmarks.items():
    status = "PASS" if low <= value <= high else "OUTSIDE RANGE"
    print(f"{metric_name:22s}: {value:.4f} -> {status}")

## Pipeline Summary

1. Downloaded the Telco Customer Churn dataset from Kaggle.
2. Cleaned `TotalCharges` with `pd.to_numeric(..., errors='coerce')` and filled missing values with `0.0`.
3. Dropped `customerID`.
4. Encoded `Churn` as `1/0`.
5. Converted categorical columns with `pd.get_dummies(..., drop_first=True)`.
6. Performed a stratified 80/20 train/test split.
7. Fitted `StandardScaler` only on training data for `tenure`, `MonthlyCharges`, and `TotalCharges`.
8. Built the required Keras `Sequential` binary classifier.
9. Trained for 50 epochs with batch size 32 and a 20% validation split.
10. Evaluated test accuracy, precision, recall, ROC-AUC, confusion matrix, and classification report.
11. Plotted training/validation loss curves.
12. Examined feature–churn correlations to identify customer factors associated with churn risk.